# OOP Week 11 -- Package Hygiene

**Course:** Object-Oriented Programming (Year 2)
**Session:** 3 hours
**Prerequisites:** Weeks 1-10
**Focus:** module boundaries, __init__.py, clean imports

---

## Learning Objectives

1. Organize code into proper Python packages
2. Write `__init__.py` files that control public API
3. Use relative imports within a package
4. Understand module boundaries and why they matter
5. Create a clean import experience for users

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Section 1: Module vs Package

| Term | What it is | Example |
|------|-----------|--------|
| **Module** | A single `.py` file | `cleaner.py` |
| **Package** | A directory with `__init__.py` | `core/` |
| **Sub-package** | A package inside a package | `core/strategies/` |

A package is just a directory that Python treats as a namespace.

---
## Section 2: Project Structure

In [ ]:
structure = """
src/
  project_name/
    __init__.py            # Top-level public API
    config.py              # Configuration handling
    core/
      __init__.py          # Export components
      data_source.py       # DataSource class
      dataset.py           # Dataset class
      cleaner.py           # BaseCleaner, RangeCleaner, etc.
      analyzer.py          # AnalyzerBase, MeanAnalyzer, etc.
      plotter.py           # Plotter class
      reporter.py          # Reporter class
      exceptions.py        # PipelineError hierarchy
      factory.py           # AnalyzerFactory
    strategies/
      __init__.py          # Export strategies
      cleaning.py          # Cleaning strategies
      analysis.py          # Analysis strategies
tests/
  __init__.py
  conftest.py              # Shared fixtures
  test_data_source.py
  test_cleaner.py
  test_analyzer.py
"""
print(structure)

**Expected Output:**
```
(project structure printed)
```

---
## Section 3: The `__init__.py` File

`__init__.py` controls what gets exported when someone writes `from project_name.core import ...`. It is the **public API** of your package.

In [ ]:
# Example: core/__init__.py
init_content = '''
"""Core pipeline components."""

from .data_source import DataSource
from .dataset import Dataset
from .cleaner import BaseCleaner, RangeCleaner, MissingCleaner
from .analyzer import AnalyzerBase, MeanAnalyzer, StdAnalyzer
from .plotter import Plotter
from .reporter import Reporter
from .exceptions import PipelineError, SchemaError, CleaningError, EmptyDataError
from .factory import AnalyzerFactory

__all__ = [
    "DataSource", "Dataset",
    "BaseCleaner", "RangeCleaner", "MissingCleaner",
    "AnalyzerBase", "MeanAnalyzer", "StdAnalyzer",
    "Plotter", "Reporter",
    "PipelineError", "SchemaError", "CleaningError", "EmptyDataError",
    "AnalyzerFactory",
]
'''
print(init_content)
print("__all__ tells Python what to export with 'from core import *'")

---
## Section 4: Clean Import Patterns

**Good imports:**
```python
from project_name.core import DataSource, Cleaner, Analyzer
from project_name.core.exceptions import SchemaError
```

**Bad imports:**
```python
from project_name.core.data_source import DataSource  # too specific
import project_name.core.cleaner  # exposes internals
from project_name.core import *  # grabs everything
```

The `__init__.py` lets users write clean imports without knowing which file each class lives in.

---
### Try It!

Write an `__init__.py` for a `strategies/` package that exports `DropMissing`, `DropOutOfRange`, and `DropDuplicates`.

In [ ]:
# YOUR CODE HERE


---
### Design Decision: What goes in __init__.py?

Only export what **users of the package** need. Internal helpers, utility functions, and base classes that are only used within the package should NOT be in `__init__.py`.

Think of `__init__.py` as the **front door** of your package. Visitors see a clean lobby, not the messy back office.

---
## Section 5: Relative Imports

In [ ]:
# Inside a package, use RELATIVE imports

# In core/cleaner.py:
# from .dataset import Dataset        # from same package
# from .exceptions import PipelineError  # from same package

# In core/__init__.py:
# from .cleaner import BaseCleaner    # from same package

# NEVER use absolute paths to import siblings:
# BAD: from project_name.core.dataset import Dataset
# GOOD: from .dataset import Dataset

print("Relative import rules:")
print("  . = current package")
print("  .. = parent package")
print("  .module = sibling module")
print("  ..other = uncle module")

**Expected Output:**
```
Relative import rules:
  . = current package
  .. = parent package
  .module = sibling module
  ..other = uncle module
```

---
## Section 6: Common Import Mistakes

---
### Common Mistake: Circular imports

The code below has a bug. Can you spot it before reading the fix?

In [ ]:
# BAD: circular dependency
# In cleaner.py: from .analyzer import Analyzer
# In analyzer.py: from .cleaner import Cleaner
# -> ImportError: cannot import name 'Analyzer'

print("Circular import: A imports B, B imports A")
print("Python cannot resolve this!")

**What goes wrong:** Circular imports happen when two modules import each other. The best fix is to restructure: extract shared classes into a separate module that both can import.

**The fix:**

In [ ]:
# FIXES for circular imports:

# Fix 1: Import inside the function that needs it
# def my_method(self):
#     from .analyzer import Analyzer
#     ...

# Fix 2: Restructure so the shared type is in a separate module
# Put Dataset in dataset.py, import it from both cleaner.py and analyzer.py

# Fix 3: Use __init__.py to control import order

print("Best fix: restructure to eliminate circular dependency")

---
### Try It!

Design the `__init__.py` for a `strategies/` sub-package that contains `cleaning.py` and `analysis.py`. What should be exported? What should stay internal?

In [ ]:
# YOUR CODE HERE


---
### Try It!

Sketch (as comments) the ideal file structure for your track's project. Include `src/`, `tests/`, and `notebooks/`. Mark which files have `__init__.py`.

In [ ]:
# YOUR CODE HERE


---
### Design Decision: Flat vs nested package structure

**Flat** (few modules, simple project):
```
src/project/
    __init__.py
    loader.py
    cleaner.py
    analyzer.py
```

**Nested** (many modules, complex project):
```
src/project/
    __init__.py
    core/
        __init__.py
        ...
    strategies/
        __init__.py
        ...
```

**Rule of thumb:** Start flat. Only nest when a directory has more than ~8 files or when you have clear sub-domains.

---
## Build from Scratch Exercise

This exercise tests whether you truly understand this week's concepts. Complete it without looking at the examples above.

In [ ]:
# BUILD FROM SCRATCH:
# Design the package structure for a weather app: data loading, cleaning, analysis, visualization. Write all __init__.py files. Show the import statements users would use.

# YOUR CODE HERE


In [ ]:
# TEST your build-from-scratch code:

# YOUR TESTS HERE


---
## Connect the Dots

How does this week's concept connect to previous weeks?

In [ ]:
# Organize your Weeks 1-10 code into a proper package structure.

# YOUR ANSWER (as comments or code):


---
## Real-World Spotting

OOP patterns are everywhere in real software. Can you spot them?

In [ ]:
# Run `import this` in Python. How do the 'Zen of Python' principles relate to package hygiene?

# YOUR ANSWER:


---
## Diagram It

Draw an ASCII class diagram for the main classes from this week. Include:
- Class names
- Key attributes
- Key methods
- Relationships (has-a, is-a)

In [ ]:
# Draw your ASCII diagram here:
# +------------------+
# |   ClassName      |
# +------------------+
# | - attribute      |
# +------------------+
# | + method()       |
# +------------------+

# YOUR DIAGRAM:


---
## Key Vocabulary

| Term | Definition |
|------|------------|
| **Module** | A single .py file |
| **Package** | A directory with __init__.py |
| **`__init__.py`** | File that makes a directory a package and controls exports |
| **`__all__`** | List of names exported by `from package import *` |
| **Relative import** | Import using `.` notation within a package |
| **Circular import** | Two modules importing each other (causes errors) |

---
## Recap Exercise

Without looking at the code above, try to:

In [ ]:
# 1. Write one class from this week FROM MEMORY
#    (it does not need to be perfect)

# YOUR CODE HERE


# 2. Create an instance and call at least one method

# YOUR CODE HERE


# 3. Write one test for your class

# YOUR CODE HERE


---
## What to Review Before Next Week

Before the next session, make sure you can:

1. Explain this week's main concept in your own words
2. Write a simple example from memory
3. Identify this pattern in existing code
4. Explain WHY this pattern is useful (not just HOW)

---
## Mini-Quiz

In [ ]:
# Q1: What is __init__.py for?
# Answer: 

# Q2: What is the difference between a module and a package?
# Answer: 

# Q3: What does __all__ control?
# Answer: 

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)